<a href="https://colab.research.google.com/github/laoneya31/IAT360-Group-18/blob/main/notebooks/Philip_Dataset_LearningRate_FinalModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1><b>Dataset Build, Learning Rate and Final Model</b></h1></center>
<center>IAT 360 Group 18 (Philip)</center>

**Tasks:** build the YOLO dataset, tune `lr0`, combine the best values into the final model.

**Hand-off:** the team is blocked on this. Upload `fruit_freshness.zip` to `IAT360-Group 18/dataset/` and everyone's notebook picks it up automatically.

# 1. Environment

Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**. The shared Drive folder must be in your My Drive: right-click `IAT360-Group 18` in *Shared with me* and choose **Organize > Add shortcut**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone our repo and the official YOLO repo. YOLO is pinned to one release so all four members train with identical code.

In [ ]:
import os, sys
REPO = "/content/IAT360-Group-18"
YOLO_TAG = "v8.4.204"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/laoneya31/IAT360-Group-18 {REPO}
# The folder must not be called "ultralytics": /content is on sys.path, so it would hide the real package.
if not os.path.exists("/content/yolo_repo"):
    !git -c advice.detachedHead=false clone -q --depth 1 --branch {YOLO_TAG} https://github.com/ultralytics/ultralytics /content/yolo_repo
%pip install -q /content/yolo_repo kagglehub
%cd {REPO}

Importing required libraries, and checking the software and hardware.

In [ ]:
import glob, hashlib, random, shutil
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

import ultralytics
ultralytics.checks()
from ultralytics import YOLO

sys.path.insert(0, f"{REPO}/src")
import fruit_yolo as fy

fy.check_config()
print("Team Drive folder:", fy.find_team_drive())
print("Baseline settings:", fy.BASELINE)

## 2. Build the dataset

Required layout (same as the course tutorial, and already in the repo):

```
datasets/fruit_freshness/
  images/train  images/val  images/test
  labels/train  labels/val  labels/test
```

**Things to watch for, found while testing the pipeline:**
- The Kaggle freshness download has a full second copy in `dataset/dataset/`. Use only `dataset/train` and `dataset/test`, never a recursive search.
- Its photos are pre-augmented screenshots (rotations of the same fruit), so split by source image before making composites. `build_composites` does this for one source folder.
- The Roboflow export may spell classes `rotton_*`. Ids still line up if the order is unchanged; `fy.check_config` confirms it.
- Use bounding-box export, not polygons.

In [ ]:
import glob, kagglehub, shutil
import xml.etree.ElementTree as ET

freshness = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
detection = kagglehub.dataset_download("mbkinaci/fruit-images-for-object-detection")
print(freshness, detection)

### Roboflow dataset

Add your API key in Colab's **Secrets** panel (key icon) as `ROBOFLOW_API_KEY`. Do not paste it in a cell, this repo is public.

In [ ]:
# TODO: fill in the version number, then convert its labels to our six classes.
# %pip install -q roboflow
# from google.colab import userdata
# from roboflow import Roboflow
# rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
# rf.workspace("ai-project-t3iak").project("detection-of-rotten-fruits").version(1).download("yolov8", location="/content/roboflow")

### Kaggle detection set: XML to YOLO

These boxes say `apple`, `banana` or `orange` with no freshness, so each box needs a freshness decision using the labelling rules. `FRESHNESS` is a starting assumption only.

In [ ]:
FRESHNESS = {"apple": 0, "banana": 1, "orange": 2}   # TODO: review images and relabel rotten ones to 3, 4, 5

def voc_to_yolo(xml_path):
    root = ET.parse(xml_path).getroot()
    width, height = int(root.find("size/width").text), int(root.find("size/height").text)
    if not width or not height:
        return None  # some files in this dataset have size 0: read the size from the image instead
    rows = []
    for obj in root.iter("object"):
        box = obj.find("bndbox")
        x1, y1, x2, y2 = (float(box.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
        rows.append(f"{FRESHNESS[obj.find('name').text]} {(x1 + x2) / 2 / width:.6f} {(y1 + y2) / 2 / height:.6f} "
                    f"{(x2 - x1) / width:.6f} {(y2 - y1) / height:.6f}")
    return rows

# TODO: loop over the XML files and write images and .txt labels into fy.DATASET.

### Composite scenes

In [ ]:
folders = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]
sources = {i: glob.glob(f"{freshness}/dataset/train/{name}/*") for i, name in enumerate(folders)}
# TODO: choose the counts. Composites go in train and val only: test is our own 120 photos.
# fy.build_composites(sources, fy.DATASET, {"train": 3000, "val": 600}, seed=0)

### Our own 120 photos

Copy all members' exported images and labels into `images/test` and `labels/test`. Faces blurred first. Export from Roboflow with auto-orient on, so EXIF rotation is already applied: `validate_dataset` warns if it is not.

In [ ]:
# TODO: copy the four members' Roboflow exports here.

### Validate and share

In [ ]:
report = fy.validate_dataset()
fy.assert_valid(report)

In [ ]:
team = fy.find_team_drive()
(team / "dataset").mkdir(exist_ok=True)
archive = shutil.make_archive("/content/fruit_freshness", "zip", fy.DATASET)
shutil.copy(archive, team / "dataset" / fy.DATASET_ZIP_NAME)
print("Shared:", team / "dataset" / fy.DATASET_ZIP_NAME)

## Hyperparameter tuning

`lr0`, baseline 0.001. This only has an effect because `BASELINE` sets the optimizer explicitly. In a 15-epoch practice run, `lr0=0.01` collapsed training with AdamW (mAP@0.5:0.95 of 0.07 against 0.80 at 0.001), so consider a lower third value such as 0.002. Tell the group if you change it.

Only this one value changes. Everything else stays at the shared baseline, so the difference from the baseline can be attributed to it.

In [ ]:
assert fy.fetch_dataset(), "Dataset is not on Drive yet."
OWNER = "Philip"
PARAM, VALUES = fy.TUNING_PLAN[OWNER]
RUNS = [f"{PARAM}_{v}" for v in VALUES]

for name, value in zip(RUNS, VALUES):
    weights = fy.train_run(name, OWNER, overrides={PARAM: value})
    fy.evaluate(name, weights)

In [ ]:
table = fy.compare_runs(names=["baseline"] + RUNS)
display(table)
fy.plot_comparison(table, f"{PARAM}: mAP@0.5:0.95 on the validation set",
                   save_to=fy.output_root() / "figures" / f"tuning_{PARAM}.png")
plt.show()
for name in RUNS:
    print(name, fy.fit_report(name))

### My results

*TODO: which value won, by how much, and why you think that happened. Copy this into the report.*

Make a zip of your runs for GitHub: unzip it at the top of the repo and commit.

In [ ]:
fy.package_results(RUNS)

## Final model

Combine the best value from each member, retrain, and evaluate. Pick values from the group's comparison, not only the single highest number.

In [ ]:
best = {"lr0": None, "imgsz": None, "batch": None, "freeze": None}   # TODO: fill in; delete any that did not beat the baseline
best = {k: v for k, v in best.items() if v is not None}
final_weights = fy.train_run("final", "Philip", overrides=best)
final = fy.evaluate("final", final_weights)
display(fy.compare_runs(names=["baseline", "final"]))